# Starpin Word Lab — Phase 0 ERG cross-check

Runs every line of `phase0_batch.jsonl` through **ERG + ACE** and records, for up to five readings per item, whether ERG produced a *clean witness*:

- every input word is covered by its own lexical terminal (no merged spans such as two `can`s collapsing into one), and
- no generic or unknown-word lexical entry was needed.

Nothing here asks anyone to judge sentences. Download `erg_phase0_results.jsonl` at the end and pass it back; `node erg/analyse.js` aggregates it by construction family and corruption type.

**Run order:** Runtime → Run all. You will be asked to upload the batch file (and the grammar image if no URL is set).

## 1. Configuration

Use the **same ACE build and ERG 2025 grammar image** as the 19 October run, so results are comparable. If you leave `ERG_DAT_URL` empty, cell 3 asks you to upload the `.dat` (or `.dat.bz2`) file.

In [ ]:
ACE_TGZ_URL = "https://sweaglesw.org/linguistics/ace/download/ace-0.9.34-x86-64.tar.gz"  # same build as 19 Oct
ERG_DAT_URL = ""        # paste the ERG 2025 grammar-image URL you used on 19 Oct, or leave empty to upload
READINGS_PER_ITEM = 5   # n-best readings inspected per item
TIMEOUT_S = 20          # per-item ACE timeout (seconds)

In [ ]:
# 2. Install PyDelphin and ACE
import subprocess, glob, os
subprocess.run(["pip", "-q", "install", "pydelphin"], check=True)
if not glob.glob("/content/ace*/ace"):
    subprocess.run(["wget", "-q", ACE_TGZ_URL, "-O", "/content/ace.tgz"], check=True)
    subprocess.run(["tar", "xzf", "/content/ace.tgz", "-C", "/content"], check=True)
ACE_BIN = glob.glob("/content/ace*/ace")[0]
print("ACE:", ACE_BIN, subprocess.run([ACE_BIN, "-V"], capture_output=True, text=True).stderr.strip()[:120])

In [ ]:
# 3. Grammar image
import shutil
GRM = "/content/erg.dat"
if not os.path.exists(GRM):
    if ERG_DAT_URL:
        target = "/content/erg.dat.bz2" if ERG_DAT_URL.endswith(".bz2") else GRM
        subprocess.run(["wget", "-q", ERG_DAT_URL, "-O", target], check=True)
    else:
        from google.colab import files
        up = files.upload()
        name = next(iter(up))
        target = "/content/erg.dat.bz2" if name.endswith(".bz2") else GRM
        shutil.move(name, target)
    if target.endswith(".bz2"):
        subprocess.run(["bunzip2", "-f", target], check=True)
print("grammar image:", GRM, round(os.path.getsize(GRM) / 2**20), "MiB")

In [ ]:
# 4. Batch file (from `node erg/prepare.js`)
import json
if not os.path.exists("/content/phase0_batch.jsonl"):
    from google.colab import files
    up = files.upload()
    shutil.move(next(iter(up)), "/content/phase0_batch.jsonl")
batch = [json.loads(l) for l in open("/content/phase0_batch.jsonl") if l.strip()]
print(len(batch), "items")

In [ ]:
# 5. Parse
import time, re
from delphin import ace

GENERIC = re.compile(r"generic|unknown|_u_unk", re.I)

def words_of(text):
    return re.findall(r"[A-Za-z']+", text)

def inspect(result, n_words):
    out = {"lexical_entities": [], "terminal_forms": [], "generic": False, "words_covered": 0, "error": None}
    try:
        d = result.derivation()
        pre = d.preterminals()
        terms = d.terminals()
        out["lexical_entities"] = [p.entity for p in pre]
        out["terminal_forms"] = [t.form for t in terms]
        out["words_covered"] = sum(len(words_of(t.form)) for t in terms)
        out["multiword_terminals"] = [t.form for t in terms if len(words_of(t.form)) > 1]
        out["generic"] = any(GENERIC.search(e or "") for e in out["lexical_entities"])
    except Exception as e:
        out["error"] = repr(e)[:200]
    mrs = result.get("mrs") or ""
    if "_u_unknown" in mrs:
        out["generic"] = True
    out["one_terminal_per_word"] = out["words_covered"] == n_words and not out.get("multiword_terminals")
    out["clean_witness"] = (not out["generic"]) and out["words_covered"] == n_words and out["error"] is None
    return out

results = []
t_start = time.time()
cmd = ["-n", str(READINGS_PER_ITEM), f"--timeout={TIMEOUT_S}", "--max-chart-megabytes=1200", "--max-unpack-megabytes=1500"]
with ace.ACEParser(GRM, executable=ACE_BIN, cmdargs=cmd, stderr=subprocess.DEVNULL) as parser:
    for i, item in enumerate(batch):
        n_words = len(words_of(item["text"]))
        rec = {"id": item["id"], "text": item["text"], "n_words": n_words}
        t0 = time.time()
        try:
            resp = parser.interact(item["text"])
            rs = resp.results()
            rec["readings"] = len(rs)
            rec["inspected"] = [inspect(r, n_words) for r in rs[:READINGS_PER_ITEM]]
            rec["first_derivation"] = rs[0].get("derivation") if rs else None
            rec["first_mrs"] = rs[0].get("mrs") if rs else None
            rec["ace_error"] = None
        except Exception as e:
            rec["readings"] = None
            rec["ace_error"] = repr(e)[:300]
        rec["ms"] = round((time.time() - t0) * 1000)
        rec["erg_parsed"] = bool(rec.get("readings"))
        rec["erg_clean"] = any(x["clean_witness"] for x in rec.get("inspected", []))
        results.append(rec)
        if i % 100 == 0:
            print(i, round(time.time() - t_start), "s")
print("done", len(results), "items in", round(time.time() - t_start), "s")

In [ ]:
# 6. Save and download
with open("/content/erg_phase0_results.jsonl", "w") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")
parsed = sum(r["erg_parsed"] for r in results)
clean = sum(r["erg_clean"] for r in results)
errors = sum(1 for r in results if r["ace_error"])
print(f"parsed {parsed}/{len(results)}, clean witness {clean}, ACE errors {errors}")
from google.colab import files
files.download("/content/erg_phase0_results.jsonl")